In [0]:
%run ./transformers/transformMap

In [0]:
from pyspark.sql.functions import *
import delta

In [0]:
class DeltaTableCreator:
    def __init__(self, spark):
        self.spark = spark

    def create_table_with_cdf(
        self,
        df: DataFrame,
        catalog: str,
        schema: str,
        table: str,
        path: str,
        #partition_cols: list = [],
        mode: str = "error"  # or 'overwrite'
    ):
        full_table_name = f"{catalog}.{schema}.{table}"

        # Escreve os dados no caminho delta com CDF ativado
        (
            df.write.format("delta")
            .mode(mode)
            .option("overwriteSchema", "true")
            .option("delta.enableChangeDataFeed", "true")
            .save(path)
        )

        # Cria a tabela no metastore apontando para o caminho
        #partition_stmt = f"PARTITIONED BY ({', '.join(partition_cols)})" if partition_cols else ""
        
        ddl = f"""
        CREATE TABLE IF NOT EXISTS {full_table_name}
        USING DELTA
        LOCATION '{path}'
        TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
        """.strip()

        self.spark.sql(ddl)
        print(f"✅ Tabela {full_table_name} criada com CDF ativado e dados em: {path}")

In [0]:
import delta

class Ingestor:
    def __init__(self, spark, source_path, data_format, target_path, catalog, schemaname, tablename):
        self.spark = spark
        self.source_path = source_path
        self.data_format = data_format
        self.target_path = target_path
        self.catalog = catalog
        self.schemaname = schemaname
        self.tablename = tablename
    
    def load(self, source_path):
        df = (
            spark.read.format(self.data_format) \
                .load(source_path)
        )
        return df

    def save(self, df):
        (df.write
         .format("delta")
         .option("path", self.target_path)
         .option("overwriteSchema", "true")
         .mode("overwrite")
         .saveAsTable(f"{self.catalog}.{self.schemaname}.{self.tablename}")
         )
        return True
    
    def executeLoadAndSave(self, path):
        df = self.load(path)
        return self.save(df)
    
class IncrementalIngestor(Ingestor):
    def __init__(self, spark, source_path, data_format, target_path, catalog, schemaname, tablename, schema_location,checkpoint_location, id_field, timestamp_field):
        super().__init__(spark, source_path, data_format, target_path, catalog, schemaname, tablename)
        self.timestamp_field = timestamp_field
        self.id_field = id_field
        self.checkpoint_location = checkpoint_location
        self.schema_location = schema_location
        self.set_deltatable()
        

    def set_deltatable(self):
        table = f"{self.catalog}.{self.schemaname}.{self.tablename}"
        self.delta_table = delta.DeltaTable.forName(self.spark, table)

    def upsert(self, df):
        
        df.createOrReplaceGlobalTempView(f"view_{self.tablename}")

        if self.timestamp_field == "N/A":
            query = f"""
        SELECT * 
        from global_temp.view_{self.tablename}
        """

        else:
            query = f"""
            SELECT * 
            from global_temp.view_{self.tablename}
            QUALIFY ROW_NUMBER() OVER(PARTITION BY {self.id_field} ORDER BY {self.timestamp_field} DESC) = 1
            """
            
        query_incremental = self.spark.sql(query)

        print("executou upsert")

        (
            self.delta_table.alias("t") \
                .merge(
                    query_incremental.alias("s"),
                    f"s.{self.id_field} = t.{self.id_field}"
                ) \
                .whenMatchedUpdateAll() \
                .whenNotMatchedInsertAll() \
                .execute()
        )

    def load(self, source_path):
       
        df = (
            spark.readStream.format("cloudFiles") \
                .option("cloudFiles.Format", self.data_format) \
                .option('cloudFiles.inferColumnTypes', 'true')
                .option("cloudFiles.schemaLocation", self.schema_location) \
                .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
                .load(source_path)
        )

        print("executou load")

        return df
    
    def save(self, df):
        print("executou save")
        stream = (df.writeStream
         .format("delta")
         .option("checkpointLocation", self.checkpoint_location)
         .option("path", self.target_path)
         .foreachBatch(lambda df, batchId: self.upsert(df))
         .trigger(availableNow=True)
        )
        return stream.start()

    def executeLoadAndSave(self, path):
        print("executou load and save")
        df = self.load(path)
        return self.save(df)

class IngestionCDF:
    def __init__(self, spark, data_format, source_schema, source_table, target_path, catalog, schemaname, tablename, schema_location,checkpoint_location, id_field, timestamp_field):
        #super().__init__(spark, data_format, target_path, catalog, schemaname, tablename)
        self.spark = spark
        self.data_format = data_format
        self.target_path = target_path
        self.catalog = catalog
        self.schemaname = schemaname
        self.source_table = source_table
        self.source_schema = source_schema
        self.tablename = tablename
        self.timestamp_field = timestamp_field
        self.id_field = id_field
        self.checkpoint_location = checkpoint_location
        self.schema_location = schema_location

    def fullLoadSilver(self, table):

        transformer = TRANSFORM_FUNCTIONS.get(table)

        bronzeSelect = f"""
            select * from {self.catalog}.{self.source_schema}.{self.source_table}
        """
        bronzeInfo = spark.sql(bronzeSelect)

        bronzeTransformed = transformer(bronzeInfo, self.spark) if transformer else bronzeInfo

        (
            bronzeTransformed.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .option("delta.enableChangeDataFeed", "true")
            .option("path", self.target_path)
            .saveAsTable(f"{self.catalog}.{self.schemaname}.{self.tablename}") 
        )

        return True


    # def load(self, source_path):
       
    #     df = (
    #         spark.readStream.format("cloudFiles") \
    #             .option("cloudFiles.Format", self.data_format) \
    #             .option('cloudFiles.inferColumnTypes', 'true')
    #             .option("cloudFiles.schemaLocation", self.schema_location) \
    #             .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    #             .load(source_path)
    #     )

    #     print("executou load")

    #     return df
    
    # def save(self, df):
    #     print("executou save")
    #     stream = (df.writeStream
    #      .format("delta")
    #      .option("checkpointLocation", self.checkpoint_location)
    #      .option("path", self.target_path)
    #      .foreachBatch(lambda df, batchId: self.upsert(df))
    #      .trigger(availableNow=True)
    #     )
    #     return stream.start()

    # def executeLoadAndSave(self, path):
    #     print("executou load and save")
    #     df = self.load(path)
    #     return self.save(df)
